In [1]:
import pandas as pd

df = pd.read_csv("/home/mpc/github/eicat-ai/data/gisd.csv")

In [2]:
groupings = df.groupby(['Reference', 'Species'])

In [6]:
import re
from pathlib import Path
from eicat_ai.models import SpeciesNames

base_path: Path = Path("evluation-set")

def sanitize_filename(name):
    return re.sub(r'[<>:"/\\|?*]', '_', name)

for name, group in groupings:
    if len(group) > 1:
        dir_name = sanitize_filename(re.split(r"[ ,]+", name[0])[0])
        path: Path = base_path / dir_name
        if path.exists():
            continue
        path.mkdir(exist_ok=True, parents=True)
        csv_file_path = path / "gold-impacts.csv"
        group.to_csv(csv_file_path, index=False)
        reference_file_path = path / "reference.txt"
        with open(reference_file_path, 'w', encoding='utf-8') as f:
            f.write(name[0])
        names: SpeciesNames = SpeciesNames(scientific_name=name[1], vernacular_names=[])
        species_names_file_path = path / "species.json"
        names.save(str(species_names_file_path))